# Diabetes Risk Prediction: A Machine Learning Research Project
**Goal:** Predict whether a patient has diabetes from basic clinical measurements (Pima Indians Diabetes dataset, 768 patients).

**Research questions**
1. Which clinical measurements are most associated with diabetes?
2. How should we handle impossible zero values (e.g. blood pressure = 0)?
3. Which model is best, is it well calibrated, and how does it behave at different risk thresholds?

*Educational project only, not for clinical use.*

## 1. Setup

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, roc_auc_score, precision_recall_curve)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
sns.set_theme(style="whitegrid")
RS = 42

## 2. Load data
Downloaded from OpenML (needs internet, which Colab has).

In [ ]:
raw = fetch_openml("diabetes", version=1, as_frame=True, parser="auto")
df = raw.frame.copy()
df.columns = ["pregnancies","glucose","blood_pressure","skin_thickness","insulin","bmi","pedigree","age","outcome"]
df["outcome"] = (df["outcome"] == "tested_positive").astype(int)   # 1 = diabetes
print(df.shape); print(df["outcome"].value_counts(normalize=True).round(3))
df.head()

## 3. Data quality
In this dataset, a value of **0** for glucose, blood pressure, skin thickness, insulin or BMI is biologically impossible. It means *missing*. We convert those zeros to NaN and impute inside the model pipeline (so no information leaks from the test set).

In [ ]:
zero_as_missing = ["glucose","blood_pressure","skin_thickness","insulin","bmi"]
print("Zeros per column (before):"); print((df[zero_as_missing] == 0).sum())
df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)
print("\nMissing % per column:"); print((df.isna().mean()*100).round(1))

## 4. Exploratory analysis

In [ ]:
feats = [c for c in df.columns if c != "outcome"]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), feats):
    sns.kdeplot(data=df, x=col, hue="outcome", fill=True, common_norm=False, ax=ax, palette=["#4C9F70","#D1495B"])
    ax.set_title(col)
    if ax.legend_: ax.legend_.remove()
plt.suptitle("Distributions by outcome (red = diabetes)", y=1.02); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix"); plt.show()
df.corr()["outcome"].drop("outcome").sort_values(ascending=False)

## 5. Split

In [ ]:
X = df.drop(columns="outcome"); y = df["outcome"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)
print(X_train.shape, X_test.shape)

## 6. Model comparison (5-fold stratified CV)
Classes are imbalanced (about 35% positive), so we use `class_weight='balanced'` where supported and look at recall and ROC AUC, not just accuracy.

In [ ]:
imp = ("imp", SimpleImputer(strategy="median"))
models = {
 "Logistic Regression": Pipeline([imp, ("sc", StandardScaler()), ("m", LogisticRegression(max_iter=5000, class_weight="balanced"))]),
 "SVM (RBF)":           Pipeline([imp, ("sc", StandardScaler()), ("m", SVC(probability=True, class_weight="balanced", random_state=RS))]),
 "Random Forest":       Pipeline([imp, ("m", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RS))]),
 "Gradient Boosting":   Pipeline([imp, ("m", GradientBoostingClassifier(random_state=RS))]),
}
cv = StratifiedKFold(5, shuffle=True, random_state=RS)
scoring = ["accuracy","precision","recall","f1","roc_auc"]
rows = []
for n, m in models.items():
    r = cross_validate(m, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({"model": n, **{s: r[f"test_{s}"].mean() for s in scoring}})
results = pd.DataFrame(rows).set_index("model").round(3).sort_values("roc_auc", ascending=False)
results

In [ ]:
results.plot(kind="bar", figsize=(11,5), ylim=(0.5,1)); plt.xticks(rotation=15)
plt.title("Cross-validated performance"); plt.legend(loc="lower right"); plt.show()

## 7. Tune the best candidate
We tune Logistic Regression (simple, interpretable). Swap in any model from above.

In [ ]:
gs = GridSearchCV(models["Logistic Regression"], {"m__C": [0.01, 0.1, 1, 10]}, cv=cv, scoring="roc_auc")
gs.fit(X_train, y_train)
best = gs.best_estimator_
print("Best:", gs.best_params_, "| CV AUC:", round(gs.best_score_, 3))

## 8. Final test-set evaluation

In [ ]:
y_prob = best.predict_proba(X_test)[:,1]
y_pred = (y_prob >= 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=["no diabetes","diabetes"]))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 3))
fig, ax = plt.subplots(1, 2, figsize=(12,5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=["no","yes"]).plot(ax=ax[0], cmap="Blues", colorbar=False)
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax[1].plot(fpr, tpr, lw=2); ax[1].plot([0,1],[0,1],"--",c="gray"); ax[1].set(title="ROC curve", xlabel="FPR", ylabel="TPR")
plt.tight_layout(); plt.show()

## 9. Is the model well calibrated?
If the model says 70% risk, do about 70% of those patients actually have diabetes?

In [ ]:
frac_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=6)
plt.figure(figsize=(5,5)); plt.plot(mean_pred, frac_pos, "o-", label="Model"); plt.plot([0,1],[0,1],"--",c="gray", label="Perfect")
plt.xlabel("Predicted risk"); plt.ylabel("Observed rate"); plt.legend(); plt.title("Calibration"); plt.show()

## 10. Threshold analysis (screening trade-off)

In [ ]:
for t in [0.5, 0.4, 0.3]:
    p = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    print(f"threshold {t}: caught={tp}, missed={fn}, false alarms={fp}, recall={tp/(tp+fn):.2f}")

## 11. Which measurements matter?

In [ ]:
pi = permutation_importance(best, X_test, y_test, scoring="roc_auc", n_repeats=30, random_state=RS)
pd.Series(pi.importances_mean, index=X.columns).sort_values().plot(kind="barh", figsize=(8,5), color="#D1495B")
plt.title("Permutation importance (drop in AUC)"); plt.show()

coef = pd.Series(best.named_steps["m"].coef_[0], index=X.columns).sort_values()
coef.plot(kind="barh", figsize=(8,5)); plt.title("Logistic regression coefficients (standardized)"); plt.show()

## 12. Conclusions & limitations
- Write your own findings here using the numbers above (best model, AUC, recall, top features).
- **Limitations:** 768 patients, all women of Pima heritage aged 21+, so results may not generalize; many missing values were imputed; no external validation.
- **Ethics:** screening models can be less accurate for groups not in the training data. Always involve clinicians.
- **Next steps:** try SHAP, XGBoost, cost-sensitive thresholds, or a different dataset (e.g. NHANES).

## References
- Smith et al. (1988), Pima Indians Diabetes Database, UCI/OpenML.
- scikit-learn documentation.